In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes.network import deepNN
from neuro_bes.preprocessing import profile_transform
from neuro_bes.data import besInferenceDatapoints


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf

In [ ]:
path="/home/molnarbalazs/data/BES_ML_modelling/W7X_op23/test_newcuration"
file_list=os.listdir(path)
file_list=[i for i in file_list if "_we" in i]

In [ ]:
len(file_list)

In [ ]:

batch=[]
for db_file in file_list:
    print(db_file)
    bes_data=besInferenceDatapoints(path=os.path.join(path,db_file))
    if bes_data.densities.shape[0]>1:
        if "we_20250514.076" in bes_data.ID:
            batch.append(bes_data)

In [ ]:
len(batch)

In [ ]:
ds=profile_transform.DensityScaler()
batch_train, batch_test = batch[:500], batch[500:]

In [ ]:
scaled_batch_train = ds.fit_transform(batch_train)
scaled_batch_test = ds.transform(batch_test)

In [ ]:
batch_train[0].densities.shape

In [ ]:
print(ds.global_max_)

In [ ]:
plt.plot(batch_train[0].grid,batch_train[0].densities.T)

In [ ]:
plt.plot(scaled_batch_train[0].grid,scaled_batch_train[0].densities.T)

In [ ]:
plt.plot(scaled_batch_train[0].grid,ds.inverse_transform(scaled_batch_train)[0].densities.T)

In [ ]:
bis=profile_transform.BeamIntensityScaler()
scaled_batch_train_bis = bis.fit_transform(batch_train[0:10])
scaled_batch_test_bis = bis.transform(batch_test[0:10])

In [ ]:
plt.subplot(1,2,1)
plt.plot(batch_train[0].grid,batch_train[0].emissions.T)
plt.subplot(1,2,2)
plt.plot(scaled_batch_train_bis[0].grid,scaled_batch_train_bis[0].emissions.T)

In [ ]:
batch_train[0].ID

In [ ]:
es=profile_transform.EmissionScaler()
scaled_batch_train_es = es.fit_transform(batch_train)
scaled_batch_test_es = es.transform(batch_test) 

In [ ]:
whichbatch=123
plt.plot(batch_train[whichbatch].grid,batch_train[whichbatch].emissions.T)

In [ ]:
plt.plot(scaled_batch_train_es[whichbatch].grid,scaled_batch_train_es[whichbatch].emissions.T)

In [ ]:
df=profile_transform.DensityFlatout2()
flat_batch_train = df.fit_transform(batch_train)
flat_batch_test = df.transform(batch_test)

In [ ]:
plt.plot(batch_train[whichbatch].grid,batch_train[whichbatch].densities.T)

In [ ]:
plt.plot(flat_batch_train[whichbatch].grid,flat_batch_train[whichbatch].densities.T)

In [ ]:
batch_test_reversed=batch_test[:1].copy()

In [ ]:
batch_test_reversed[0].grid=batch_test_reversed[0].grid[::-1]
batch_test_reversed[0].densities=batch_test_reversed[0].densities[:,::-1]
batch_test_reversed[0].emissions=batch_test_reversed[0].emissions[:,::-1]

In [ ]:
plt.plot(batch_test_reversed[0].grid,batch_test_reversed[0].densities.T)

In [ ]:
batch_test_reversed[0].grid

In [ ]:
flat_batch_test_reversed=df.transform(batch_test_reversed)

In [ ]:
plt.plot(flat_batch_test_reversed[0].grid,
         flat_batch_test_reversed[0].densities.T)

In [ ]:
interp_cg=profile_transform.InterpolateToCommonGrid()
interp_batch_train=interp_cg.fit_transform(batch_train)
interp_batch_test=interp_cg.transform(batch_test)

In [ ]:
plt.plot(batch_train[0].grid,batch_train[0].emissions.T)

In [ ]:
plt.plot(interp_batch_train[0].grid,interp_batch_train[0].emissions.T)

In [ ]:

pipeline = profile_transform.PartialInversePipeline([
    ('density_scaler', profile_transform.DensityScaler()),
    ('emission_scaler', profile_transform.EmissionScaler()),
    ('density_flatout', profile_transform.DensityFlatout()),
    ('interpolate_to_common_grid', profile_transform.InterpolateToCommonGrid())
])

In [ ]:
pipeline_batch_train = pipeline.fit_transform(batch_train)
pipeline_batch_test = pipeline.transform(batch_test)

In [ ]:
hasattr(pipeline, "inverse_transform")

In [ ]:
inversepipline_batch_test = pipeline.inverse_transform(pipeline_batch_test)

In [ ]:
plt.plot(batch_test[10].grid,batch_test[10].densities[2])
plt.plot(inversepipline_batch_test[10].grid,inversepipline_batch_test[10].densities[2])

In [ ]:
plt.plot(batch_test[10].grid,batch_test[10].emissions[2])
plt.plot(inversepipline_batch_test[10].grid,inversepipline_batch_test[10].emissions[2])

In [ ]:
emission=np.concatenate([batch.emissions for batch in pipeline_batch_train])
density=np.concatenate([batch.densities for batch in pipeline_batch_train])

In [ ]:
for i in (np.random.rand(100)*density.shape[0]).astype(int):
    plt.plot(density[i])

In [ ]:
for i in (np.random.rand(100)*emission.shape[0]).astype(int):
    plt.plot(emission[i])

In [ ]:
emission.shape